# 09 · Checkpoint: capacity report for one model
Level L1 · Part 1 · core · ~20 min · builds on 05, 07, 08

## Goal
By the end you have a one-page capacity report for the Brightside support model. It has two numbers and one verdict: a **concurrency estimate** for one GPU (from the memory budget, lesson 05), a **measured p95 TTFT** against a target you set (from the load method in lesson 07, on a real server from lesson 08), and a line that says whether the target holds.

## Why it matters
- **Without it:** someone asks "can one GPU take our support traffic?" and the answer is a guess, or a number from a slide. The guess can be wrong by 2 to 5 times, and nobody can check it.
- **With it:** the same four numbers (GPU memory, weights, activations, tokens per request) give a concurrency limit. A short measured run against a real server checks the latency side. Both go in one report with the target next to them, so the decision can be checked.
- **When you don't need it:** a hosted provider sets the limits for you and publishes its latency. You build this report when you run the server yourself and must choose the hardware.

## The idea
A capacity report answers two questions. **How many requests fit?** That is memory arithmetic: the KV budget divided by the memory each request needs. **How fast is the first token under load?** That is a measurement, not arithmetic: send requests at several concurrency levels and read the p95 of the time to first token (TTFT). A report with only the first number is a hope. A report with only the second is a measurement of one machine that may not be your GPU. The report needs both, and it must say which numbers were measured and which were estimated.

Think of a bridge inspection. The engineer calculates the load limit from the plans, then drives test trucks across and records the sag. The calculation says what should hold. The test says what does hold on this bridge.

## Requirements
- [ ] A concurrency estimate for **Llama 3.1 8B in FP16 on one NVIDIA L4 (24 GB)**, with 1,800 tokens per request (lesson 05).
- [ ] A **target** you choose: p95 TTFT in seconds (this notebook uses 2.0 s).
- [ ] A **measured** p95 TTFT at 1, 2 and 4 concurrent requests on the pinned vLLM CPU image, serving Qwen2.5-0.5B-Instruct (lessons 07 and 08).
- [ ] A **verdict** for each level and an overall capacity at the target, labelled as CPU-measured.
- [ ] A statement of what the numbers do not prove: the measurement is on a CPU and a smaller model, so it tests the target and the method, not the GPU.

This notebook runs on a laptop with Docker. Without Docker, the measurement cell prints **recorded output** from a live run of the same image, labelled as such.

## Build it
### Step 1: the memory estimate (lesson 05)
The same arithmetic as lesson 05, in a few lines. The GPU is not used here: this is pure arithmetic. First the imports, the constants and the Docker check.

In [1]:
import json, sys, time, asyncio
from pathlib import Path
import numpy as np
sys.path.insert(0, "..")          # the course folder, so tut_env.py can be imported
sys.path.insert(0, "../apps/09")  # lab09.py: start, wait for and stop the Docker server
from tut_env import env
import lab09

IMAGE = "vllm/vllm-openai-cpu:v0.32.0"     # pinned: the image lesson 08 used
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"       # Apache-2.0, about 1 GB
SERVED = "brightside-qwen-0.5b"            # the name the API uses
NAME = "tut-kv-forge-15h-09"               # container name, unique to this course and lesson
PORT = 8090                                # this lesson's port (8090 to 8099)
BASE = f"http://127.0.0.1:{PORT}"
TARGET_P95_TTFT_S = 2.0                    # your target: p95 time to first token, in seconds
LEVELS = (1, 2, 4)                         # concurrent requests to try (the server runs 4 at once)
N_PER_LEVEL = 12                           # requests per level
KEY = env("VLLM_API_KEY", "")              # empty when unset; the key is never printed
DATA = Path("../data/09")
DATA.mkdir(parents=True, exist_ok=True)
GIB = 2**30

DOCKER = lab09.docker_ready()          # False: the measurement cell uses the recorded rows
print("Docker available:", DOCKER)

Docker available: True


Two small functions. `kv_bytes_per_token` is lesson 03's formula. `max_concurrency` is lesson 05's budget: what is left of the GPU after weights and activations, divided by what one request needs.

In [2]:
def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_value=2):
    """Keys and values, every layer, every KV head (lesson 03)."""
    return 2 * layers * kv_heads * head_dim * bytes_per_value

def max_concurrency(gpu_bytes, weights, kv_per_token, tokens, activations=1.0 * GIB, util=0.92, max_seqs=128):
    """Requests that fit by memory, and the number that actually runs (capped by max_num_seqs)."""
    left = gpu_bytes * util - weights - activations            # the KV budget (lesson 05)
    if left <= 0:
        raise ValueError("weights do not fit with the activations in this GPU")
    by_memory = int(left // (kv_per_token * tokens))
    return min(by_memory, max_seqs), by_memory

Now the 8B numbers. The KV budget is 92% of the L4, minus the weights, minus 1 GiB of activations. Each request needs its tokens times the bytes per token. The print shows both steps, so you can check the 20 by hand.

In [3]:
KV_8B = kv_bytes_per_token(32, 8, 128)     # 128 KiB per token, FP16
WEIGHTS_8B = 8_030_261_248 * 2             # 8.03 billion parameters x 2 bytes (FP16)
TOKENS = 1_600 + 200                       # policy prompt + answer, as in lesson 05
L4_BYTES = 24e9                            # NVIDIA L4 spec: 24 GB

kv_budget = L4_BYTES * 0.92 - WEIGHTS_8B - 1.0 * GIB
per_request = KV_8B * TOKENS
print(f"KV budget: {kv_budget / GIB:.2f} GiB; per request: {per_request / 2**20:.0f} MiB "
      f"({TOKENS:,} tokens x {KV_8B // 1024} KiB). Expected: 20 fit.")
running, by_memory = max_concurrency(L4_BYTES, WEIGHTS_8B, KV_8B, TOKENS)
print(f"estimate, Llama 3.1 8B FP16 on one L4, {TOKENS:,} tokens per request: "
      f"{by_memory} fit in memory, {running} run")

KV budget: 4.61 GiB; per request: 225 MiB (1,800 tokens x 128 KiB). Expected: 20 fit.
estimate, Llama 3.1 8B FP16 on one L4, 1,800 tokens per request: 20 fit in memory, 20 run


### Step 2: start the real server (lesson 08's pinned CPU image)
The container uses port **8090**, this lesson's own port. It does not use `VLLM_BASE_URL` from `.env`, which points at port 8000. The model cache from lesson 08 is reused, so nothing is downloaded again. The first answer on a CPU is slow: lesson 08 measured about 340 s, and a later cold start on the same machine took about 650 s.

In [4]:
ARGS = [MODEL, "--served-model-name", SERVED, "--host", "0.0.0.0", "--dtype", "float32", "--enforce-eager",
        "--max-model-len", "4096", "--max-num-seqs", "4", "--kv-cache-memory-bytes", str(2 * 1024**3)]
if DOCKER:
    lab09.start_server(NAME, IMAGE, PORT, ARGS, KEY)     # the helper removes any old container first
else:
    print("Docker is not running here: the measurement will use recorded output.")

⏭  Skipped: this cell needs [docker,download]. Run it yourself, or re-run with TUT_ENABLE=docker,download.


**Wait until healthy.** This cell polls the server every 2 s, for up to 480 s, and stops early if the container dies. Expect the first answer to take 6 to 11 minutes on 4 CPU cores, so the total wait before the table is about 10 to 20 minutes.

In [5]:
if DOCKER:
    lab09.wait_healthy(NAME, BASE)
    print("server healthy on port", PORT)

⏭  Skipped: this cell needs [docker,download]. Run it yourself, or re-run with TUT_ENABLE=docker,download.


### Step 3: measure p95 TTFT at three concurrency levels (lesson 07's method)
Each request is streamed. TTFT is the time from sending the request to the first token of text. The first request is a warm-up and is not counted. The prompts are short (about 70 tokens), not the 1,800-token request of step 1: that is the honest limit of a CPU run.

In [6]:
from openai import AsyncOpenAI

QUESTIONS, SYSTEM = lab09.QUESTIONS, lab09.SYSTEM     # four support questions and the policy prompt

async def timed(client, question):
    """Seconds from sending the request to the first text token."""
    t0 = time.perf_counter()
    first = None
    stream = await client.chat.completions.create(
        model=SERVED, stream=True, max_tokens=24, temperature=0,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}])
    async for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content and first is None:
            first = time.perf_counter() - t0
    return first

The client runs each level with `c` requests at once. A semaphore is a counter of free slots: it lets only `c` requests into the server at a time, so the level you name is the concurrency the server really sees. Without it, all 12 requests would start together and the level would mean nothing.

In [7]:
async def measure():
    client = AsyncOpenAI(base_url=f"{BASE}/v1", api_key=KEY or "EMPTY", timeout=900)
    rows = []
    for c in LEVELS:
        sem = asyncio.Semaphore(c)                     # at most c requests in flight
        async def one(q):
            async with sem:
                return await timed(client, q)
        ttfts = np.array(await asyncio.gather(*(one(QUESTIONS[i % len(QUESTIONS)]) for i in range(N_PER_LEVEL))))
        rows.append({"concurrency": c, "n": len(ttfts),
                     "ttft_p50_s": round(float(np.percentile(ttfts, 50)), 3),
                     "ttft_p95_s": round(float(np.percentile(ttfts, 95)), 3)})
    await client.close()
    return rows

The first answer is slow on a CPU (roughly 6 to 11 minutes on 4 cores), so it gets its own cell, tagged `slow`. Its time is printed and it is not counted in the table. If your cell limit is shorter than that, raise it (for example `TUT_CELL_TIMEOUT=1500`) before you run the notebook.

In [8]:
if lab09.server_up(BASE):
    warm = AsyncOpenAI(base_url=f"{BASE}/v1", api_key=KEY or "EMPTY", timeout=1500)
    t0 = time.perf_counter()
    await timed(warm, QUESTIONS[0])          # warm-up, not counted
    print(f"warm-up: first answer in {time.perf_counter() - t0:.0f} s (not counted)")
    await warm.close()

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


Run the warm-up cell (the one above, tagged `slow`) first, so the cold first answer (about 600 s) is not counted in the level-1 row. Then run the measurement and print the table. Level 1 is 12 requests one at a time, level 4 is the same 12 with four in flight. The cell uses the live server if it is up, and the recorded rows if it is not. The total wait is a few minutes after the warm-up.

In [9]:
if lab09.server_up(BASE):
    rows = await measure()
    source = "live run on the pinned CPU image (this notebook)"
else:
    rows = json.loads(Path("../data/recorded/09.json").read_text())["rows"]
    source = "recorded output: a live run of this notebook on the same image, 2026-10-22"

print(f"source: {source}")
print(f"{'concurrency':>11} {'n':>3} {'TTFT p50':>9} {'TTFT p95':>9}")
for r in rows:
    print(f"{r['concurrency']:>11} {r['n']:>3} {r['ttft_p50_s']:>8.2f}s {r['ttft_p95_s']:>8.2f}s")

source: recorded output: a live run of this notebook on the same image, 2026-10-22
concurrency   n  TTFT p50  TTFT p95
          1  12     0.88s     0.99s
          2  12     1.14s     1.91s
          4  12     1.21s     3.02s


**Expected table** (recorded output, 2026-10-22): p95 TTFT is about 0.99 s at 1 request, 1.91 s at 2 and 3.02 s at 4, so the target of 2.0 s holds up to 2 requests. Your numbers will differ a little; the shape should match.

### Step 4: your turn — the capacity at your target
Write `capacity_at_target(rows, target_s)`. It returns the **highest concurrency whose p95 TTFT is at or below the target**, or 0 when none is. Use the `rows` list from step 3. Then compare your answer with the hidden answer below.

In [10]:
def capacity_at_target(rows, target_s):
    return None  # your code here

**Answer** — try it first, then expand the hidden cell below

In [11]:
def capacity_at_target(rows, target_s):
    ok = [r["concurrency"] for r in rows if r["ttft_p95_s"] <= target_s]
    return max(ok) if ok else 0

print("expected capacity_at_target(rows, 2.0):", capacity_at_target(rows, 2.0))

expected capacity_at_target(rows, 2.0): 2


### Watch out
Complete step 4 before running on: this cell calls your function with a string target. The target must be a number of seconds, not text. This is the error you get with `"2 s"` in place of `2.0`:

In [12]:
capacity_at_target(rows, "2 s")   # the fix: pass TARGET_P95_TTFT_S, a float in seconds

TypeError: '<=' not supported between instances of 'float' and 'str'

Expected error after the cell above: `TypeError: '<=' not supported between instances of 'float' and 'str'`. The comparison `r["ttft_p95_s"] <= target_s` compares a number with text. The fix is to pass `TARGET_P95_TTFT_S` (2.0), a float.

## Reference solution
The report puts the estimate, the measured rows, the target and the verdict in one place. It is also saved to `data/09/report.json`. Try the steps above before you read this cell's output.

In [13]:
cap_cpu = capacity_at_target(rows, TARGET_P95_TTFT_S)
verdict = (f"meets the target at every level tried (up to {max(LEVELS)}) on this CPU" if cap_cpu == max(LEVELS)
           else f"partial: the target holds up to {cap_cpu} concurrent request(s) on this CPU" if cap_cpu
           else "miss: even one request misses the target on this CPU")
report = {"estimate": {"gpu": "NVIDIA L4 24 GB", "tokens_per_request": TOKENS, "fit_by_memory": by_memory,
                       "running": running, "basis": "arithmetic only (lesson 05), not measured"},
          "measured": {"server": IMAGE, "model": MODEL, "source": source, "rows": rows,
                       "basis": "measured on a CPU with a 0.5B model and short prompts"},
          "target_p95_ttft_s": TARGET_P95_TTFT_S, "verdict": verdict,
          "limits": ("The CPU run tests the target and the method. It does not measure the 8B model on the L4. "
                     "The memory estimate uses 1,800 tokens per request; the measurement uses prompts of about 70 tokens.")}
(DATA / "report.json").write_text(json.dumps(report, indent=1))
print(f"measured p95 TTFT vs target {TARGET_P95_TTFT_S} s: {verdict}")
print("limits:", report["limits"])

measured p95 TTFT vs target 2.0 s: partial: the target holds up to 2 concurrent request(s) on this CPU
limits: The CPU run tests the target and the method. It does not measure the 8B model on the L4. The memory estimate uses 1,800 tokens per request; the measurement uses prompts of about 70 tokens.


**What you should see:** the first line reads `estimate, Llama 3.1 8B FP16 on one L4, 1,800 tokens per request: 20 fit in memory, 20 run`. That is the 20 from the budget (about 4.6 GiB of KV space, 225 MiB per request, 20.96 requests before rounding down). The measured p95 rises from about 1.0 s at 1 request to 3.0 s at 4, and the warm-up took about 10 minutes. The verdict is a CPU result. It says whether the method and the target work. Whether the L4 holds 20 requests at your target is a question for a GPU run, not for this notebook.

In [14]:
if DOCKER:
    lab09.save_logs_and_stop(NAME, DATA / "server.log")
    print("container removed; server log saved to data/09/server.log")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


## Recap
- The report has an estimate (arithmetic) and a measurement (a real server), with the target between them.
- The estimate for an 8B model on an L4 is 20 requests: about 4.6 GiB of KV space, at 225 MiB per 1,800-token request.
- A p95 over 12 requests per level is a sanity check. Say how many requests you measured.
- A CPU run with a small model cannot prove a GPU target. Say what was measured and on what.

**Interview:** "How do you know one GPU can serve your traffic?" — Strong answer: compute the KV budget and the concurrency limit from the memory, measure p95 TTFT under the concurrency you expect against a target you set, and report both with their basis.

## At work
- **Benchmark tools:** teams load-test with `vllm bench serve` (the current vLLM subcommand; the old `benchmark_serving.py` script is legacy) or GuideLLM 0.8.1, which is built around latency SLOs. Both send requests at a rate and report TTFT and TPOT percentiles. → lesson 27 covers realistic arrival rates and length distributions.
- **Goodput, not tokens per second:** production targets set p95 TTFT and p95 inter-token latency together. A server that is fast on average and misses the p95 target fails the SLO. → lesson 27.
- **The failure this report prevents:** a CPU run, or one request on a laptop, reported as the GPU result. Queueing then makes p95 far worse under real traffic (Little's law, lesson 26). Label the basis of every number.
- **Go deeper:** multi-replica routing and autoscaling on TTFT are in the L6 lessons.

## Key terms
- **capacity report** — a one-page record of the concurrency estimate, the measured latency, the target and the verdict (lesson 09)
- **concurrency estimate** — the number of requests that fit in the KV budget (lesson 05)
- **p95 TTFT** — the time to first token that 95% of requests beat (lessons 07, 08)
- **recorded output** — saved output from a real run, shown in place of a live run when the server cannot start (lesson 08)